Now we model **diffuse light scattering**.

Before:

```text
Ray hits object → just stop
```

Now:

```text
Ray hits object → bounce randomly
```

This is **Lambertian reflection**.

---

# 1. What is Lambertian?

Lambertian means light scatters equally in all directions.

Reflectance = **albedo**

Albedo:

$$
0 \le A \le 1
$$

where:

* $A=0$ → absorbs all light
* $A=1$ → reflects all light

Color update:

$$
C_{new} = A \cdot C_{old}
$$

Example:

If:

$$
A=(0.8,0.3,0.3)
$$

then:

$$
C_{new}=(0.8R,0.3G,0.3B)
$$

---

# 2. Scatter direction formula

We generate:

$$
S = N + U
$$

where:

* $N$ = surface normal
* $U$ = random unit vector

This creates diffuse randomness.

---

# 3. Problem: zero vector

Rare case:

$$
U = -N
$$

Then:

$$
S = N + (-N) = 0
$$

Bad:

$$
|S|=0
$$

because:

$$
\hat S = \frac{S}{|S|}
$$

division by zero.

So we add:

```python
near_zero()
```

---

# Updated Python Code

---

# material.py

```python
from abc import ABC, abstractmethod


class Material(ABC):

    """
    Base material class

    Before:
        no actual material behavior

    Now:
        child classes must define scatter()
    """

    @abstractmethod
    def scatter(self, r_in, rec):
        pass
```

---

# lambertian.py

```python
from util.material import Material
from util.ray import Ray
from util.vec3 import random_unit_vector


class Lambertian(Material):

    def __init__(self, albedo):
        """
        NEW:
        store reflectance color

        albedo:
            how much light survives after hit

        Formula:
        C_new = A * C_old
        """
        self.albedo = albedo

    def scatter(self, r_in, rec):

        """
        Lambertian diffuse formula:

        S = N + U

        where:
            N = surface normal
            U = random unit vector
        """

        # NEW:
        # generate random scatter direction
        scatter_direction = rec.normal + random_unit_vector()

        """
        CHANGED:
        Before:
            directly use scatter_direction

        Problem:
            can become zero vector

        Example:
            random_unit_vector() == -normal

        Then:
            normal + (-normal) = 0

        Fix:
            use normal directly
        """
        if scatter_direction.near_zero():
            scatter_direction = rec.normal

        """
        NEW:
        scattered ray starts at hit point
        """
        scattered = Ray(rec.p, scatter_direction)

        """
        NEW:
        attenuation is material color
        """
        attenuation = self.albedo

        """
        always scatter
        """
        return True, attenuation, scattered
```

---

# vec3.py (what changed)

Add:

```python
def near_zero(self):
    """
    NEW FUNCTION

    Detect nearly zero vector

    Formula:

    |x| < ε
    |y| < ε
    |z| < ε

    ε = 10^-8
    """

    s = 1e-8

    return (
        abs(self.x) < s and
        abs(self.y) < s and
        abs(self.z) < s
    )
```

---

# Step-by-step learning table

| Step | Code                   | Formula       | Purpose           |           
| ---- | ---------------------- | ------------- | ----------------- | 
| 1    | `self.albedo = albedo` | $A$           | store reflectance |            
| 2    | `random_unit_vector()` | $U$           | random bounce     |            
| 3    | `rec.normal + U`       | $S=N+U$       | diffuse scatter   |            
| 4    | `near_zero()`          | $S\approx 0$  | avoid invalid vector |
| 5    | `Ray(rec.p, S)`        | $R_s(t)=P+tS$ | create new ray    |            
| 6    | `attenuation=albedo`   | $C'=A\cdot C$ | dim light         |            

---

# Full flow math

Incoming ray:

$$
R(t)=O+tD
$$

Hit point:

$$
P=O+tD
$$

Normal:

$$
N=\frac{P-C}{r}
$$

Random direction:

$$
U=random_unit_vector()
$$

Scatter:

$$
S=N+U
$$

New ray:

$$
R_s(t)=P+tS
$$

Color:

$$
C_{next}=A\cdot C
$$

---

# Data flow

```text
Ray hits sphere
        ↓
Get hit_record
        ↓
Get normal
        ↓
Generate random unit vector
        ↓
Add normal + random vector
        ↓
Check near_zero
        ↓
Create scattered ray
        ↓
Apply albedo attenuation
        ↓
Continue tracing
```

Core idea:

$$
IncomingLight \rightarrow ScatterRandomly \rightarrow LoseEnergy
$$

Lambertian is the simplest realistic diffuse model.


# Lambertian

In [2]:
import sys 
sys.path.append("../../Ray Tracing in One Weekend/")

In [4]:
from util.material import Material
from util.ray import Ray
from util.vec3 import random_unit_vector


class Lambertian(Material):

    def __init__(self, albedo):
        """
        NEW:
        store reflectance color

        albedo:
            how much light survives after hit

        Formula:
        C_new = A * C_old
        """
        self.albedo = albedo

    def scatter(self, r_in, rec):

        """
        Lambertian diffuse formula:

        S = N + U

        where:
            N = surface normal
            U = random unit vector
        """

        # NEW:
        # generate random scatter direction
        scatter_direction = rec.normal + random_unit_vector()

        """
        CHANGED:
        Before:
            directly use scatter_direction

        Problem:
            can become zero vector

        Example:
            random_unit_vector() == -normal

        Then:
            normal + (-normal) = 0

        Fix:
            use normal directly
        """
        if scatter_direction.near_zero():
            scatter_direction = rec.normal

        """
        NEW:
        scattered ray starts at hit point
        """
        scattered = Ray(rec.p, scatter_direction)

        """
        NEW:
        attenuation is material color
        """
        attenuation = self.albedo

        """
        always scatter
        """
        return True, attenuation, scattered